# Challenge 04 -- Deploy a Hosted Weather Agent on Foundry

## Learning Objectives
- Define a custom tool function for a hosted agent
- Write agent instructions that control behavior
- Configure the agent definition (protocol, resources, environment)
- Build a container image and deploy to Foundry Agent Service
- Invoke the hosted agent and observe tool-calling behavior

Replace every `___` with the correct code. There are 10 blanks to fill in.

---
## Setup -- Install Dependencies and Authenticate

Run these cells as-is. No changes needed.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv pyyaml --quiet

In [ ]:
import os

PROJECT_ENDPOINT = "https://new-agent-test-ai.services.ai.azure.com/api/projects/WTH-Hosted-Agents"
MODEL_DEPLOYMENT_NAME = "gpt-4-1"
ACR_NAME = "crxutdtaalmabgq"

assert PROJECT_ENDPOINT and "/api/projects/" in PROJECT_ENDPOINT
assert MODEL_DEPLOYMENT_NAME
assert ACR_NAME

print(f"Endpoint: {PROJECT_ENDPOINT}")
print(f"Model:    {MODEL_DEPLOYMENT_NAME}")
print(f"ACR:      {ACR_NAME}")

In [ ]:
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential

TENANT_ID = os.getenv("AZURE_TENANT_ID")

credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

credential.get_token("https://management.azure.com/.default")
print("Connected to Azure")

---
## Part 1: Define the Weather Tool

Hosted agents can call **tool functions** to retrieve data. The `@tool` decorator from
the Agent Framework SDK marks a Python function as callable by the agent. When a user
asks about the weather, the agent will invoke this function automatically.

Your tool function should:
- Accept a `location` parameter (the city name)
- Return a string with weather information
- Use the `Annotated` type hint with a `Field` description so the agent knows what the parameter means

Fill in the 3 blanks below:
1. The type annotation for the `location` parameter (use `Annotated` with a `Field` description)
2. The return type of the function
3. The dictionary lookup that returns weather data for the location

In [ ]:
from typing import Annotated
from pydantic import Field

# Simulated weather data -- in production this would call a real weather API
WEATHER_DATA = {
    "seattle": "55F, cloudy with light rain",
    "new york": "72F, sunny with a few clouds",
    "london": "60F, overcast with drizzle",
    "tokyo": "80F, warm and humid",
    "paris": "63F, clear skies",
}

def get_weather(
    location: ___  # BLANK 1: Annotated[str, Field(description="...")]
) -> ___:  # BLANK 2: What type does this function return?
    '''Get the current weather for a given location.'''
    result = ___  # BLANK 3: Look up location in WEATHER_DATA (lowercase), with a default
    return f"The weather in {location} is: {result}"

# Test it right here
print(get_weather("Seattle"))
print(get_weather("Tokyo"))
print(get_weather("Narnia"))

---
## Part 2: Write Agent Instructions

The agent instructions define its persona and behavior -- just like the system prompt
you wrote in Challenge 02, but this time the instructions go into the hosted agent code.

Good instructions should tell the agent:
- What it is and what it does
- When and how to use the weather tool
- How to handle locations not in the data
- What tone to use in responses

Fill in the blank with your own instructions (at least 2-3 sentences).

In [ ]:
AGENT_INSTRUCTIONS = ___  # BLANK 4: Write a string with your agent's instructions

print("Instructions length:", len(AGENT_INSTRUCTIONS), "chars")
print()
print(AGENT_INSTRUCTIONS)

---
## Part 3: Assemble the Agent Code

Now we combine your tool function and instructions into `main.py` -- the complete
hosted agent that will run inside a container on Foundry.

The assembly cell below takes your `get_weather` function and `AGENT_INSTRUCTIONS`
and writes them into the full agent code. It also adds:
- **FoundryChatClient**: connects to your model deployment (credentials provided by the platform)
- **Agent**: combines the client, instructions, and tools
- **ResponsesHostServer**: exposes the agent as an HTTP endpoint on port 8088

Run this cell as-is -- it uses `inspect.getsource()` to capture the function you wrote above.

In [ ]:
import inspect, textwrap

# Get the source of the tool function and weather data you defined above
tool_source = inspect.getsource(get_weather)
data_source = "WEATHER_DATA = " + repr(WEATHER_DATA)

# Write the complete main.py
os.makedirs("weather-agent", exist_ok=True)

with open("weather-agent/main.py", "w") as f:
    f.write('''import os
from typing import Annotated

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from pydantic import Field

load_dotenv()


# --- Weather data and tool ---

''' + data_source + '''


@tool(approval_mode="never_require")
''' + tool_source + '''


# --- Create the Foundry chat client ---

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)


# --- Create the agent ---

agent = Agent(
    client=client,
    instructions=''' + repr(AGENT_INSTRUCTIONS) + ''',
    tools=[get_weather],
    default_options={"store": False},
)


# --- Start the hosted server ---

server = ResponsesHostServer(agent)
server.run()
''')

# Verify
import py_compile
py_compile.compile("weather-agent/main.py", doraise=True)
print("[OK] weather-agent/main.py written and syntax-valid")

# Show the result
with open("weather-agent/main.py") as f:
    for i, line in enumerate(f, 1):
        print(f"{i:3d}  {line}", end="")

---
## Part 4: Configure the Agent Definition

The `agent.yaml` file tells Foundry how to run your container:
- **kind**: `hosted` means this is a containerized agent (not prompt-based)
- **protocols**: which HTTP protocol the agent exposes (`responses` = OpenAI-compatible)
- **resources**: CPU and memory allocation for the container
- **environment_variables**: values injected at runtime (model name, etc.)

Run these cells as-is.

In [ ]:
import yaml

agent_config = {
    "kind": "hosted",
    "name": "weather-agent",
    "protocols": [
        {"protocol": "responses", "version": "1.0.0"}
    ],
    "resources": {
        "cpu": "0.5",
        "memory": "1Gi",
    },
    "environment_variables": [
        {"name": "AZURE_AI_MODEL_DEPLOYMENT_NAME", "value": MODEL_DEPLOYMENT_NAME}
    ],
}

with open("weather-agent/agent.yaml", "w") as f:
    yaml.dump(agent_config, f, default_flow_style=False, sort_keys=False)

print("[OK] weather-agent/agent.yaml written")
print()
with open("weather-agent/agent.yaml") as f:
    print(f.read())

In [ ]:
# Write Dockerfile and supporting files
with open("weather-agent/Dockerfile", "w") as f:
    f.write("""FROM python:3.12-slim

WORKDIR /app

COPY . user_agent/
WORKDIR /app/user_agent

RUN if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

EXPOSE 8088

CMD ["python", "main.py"]
""")

with open("weather-agent/requirements.txt", "w") as f:
    f.write("""agent-framework>=1.2.2
agent-framework-foundry-hosting
azure-identity
python-dotenv
pydantic
""")

with open("weather-agent/.dockerignore", "w") as f:
    f.write(""".venv
__pycache__
*.pyc
.env
""")

# Show project structure
print("[OK] All agent files written")
print()
print("weather-agent/")
for fname in sorted(os.listdir("weather-agent")):
    if not fname.startswith("__"):
        size = os.path.getsize(f"weather-agent/{fname}")
        print(f"  {fname} ({size} bytes)")

---
## Part 5: Build the Container Image

This uses `az acr build` to build the Docker image remotely on Azure Container Registry.
No local Docker installation needed -- ACR does the build in the cloud.

This typically takes about 2 minutes. Run as-is.

In [ ]:
import json as _json
import subprocess

print("Building container image on ACR (this takes about 2 minutes)...")

result = subprocess.run(
    f"az acr build --registry {ACR_NAME} --image weather-agent:v1 "
    f"--platform linux/amd64 weather-agent/ --no-logs",
    shell=True, capture_output=True, text=True,
)

if result.returncode == 0:
    build_info = _json.loads(result.stdout)
    status = build_info.get("status", "unknown")
    images = build_info.get("outputImages", [])
    print(f"Build status: {status}")
    if images:
        print(f"Image: {images[0]['registry']}/{images[0]['repository']}:{images[0]['tag']}")
    if status == "Succeeded":
        print("[OK] Container image built successfully")
    else:
        print("[WARN] Build finished but status is: " + status)
else:
    print("[FAIL] Build failed:")
    print(result.stderr[:500])

---
## Part 6: Deploy the Agent to Foundry

Use `AIProjectClient` to create an agent version that points to the container image
you just built. Foundry provisions a container and makes the agent available as a
hosted endpoint.

`HostedAgentDefinition` specifies: the container image, the protocol, CPU/memory
resources, and environment variables to inject at runtime.

Fill in the 4 blanks:
1. The protocol the agent uses (hint: OpenAI-compatible Responses API)
2. The CPU allocation (a string like "0.5")
3. The container image URL (ACR registry + image name + tag)
4. The environment variable dictionary (what does the agent need to know at runtime?)

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)

# Check if any version is already active (avoids creating duplicates on re-run)
active_version = None
try:
    for v in project.agents.list_versions(agent_name="weather-agent"):
        if v["status"] == "active":
            active_version = v
            break
except Exception:
    pass

if active_version:
    print(f"Agent weather-agent v{active_version.version} is already active -- skipping creation")
    agent_version = active_version
else:
    agent_version = project.agents.create_version(
        agent_name="weather-agent",
        definition=HostedAgentDefinition(
            container_protocol_versions=[
                ProtocolVersionRecord(protocol=___, version="1.0.0")  # BLANK 5: Which protocol?
            ],
            cpu=___,  # BLANK 6: CPU allocation as a string
            memory="1Gi",
            image=___,  # BLANK 7: Full image URL (f"{ACR_NAME}.azurecr.io/...")
            environment_variables=___,  # BLANK 8: Dict mapping env var name to value
        ),
    )
    print(f"Agent created: {agent_version.name}, version: {agent_version.version}")

In [ ]:
import time

version_num = str(agent_version.version)

while True:
    info = project.agents.get_version(agent_name="weather-agent", agent_version=version_num)
    status = info["status"]
    print(f"Status: {status}")
    if status == "active":
        print("[OK] Agent is deployed and ready")
        break
    elif status == "failed":
        print("[FAIL] Provisioning failed")
        break
    time.sleep(10)

---
## Part 7: Invoke the Deployed Agent

Send requests to the hosted agent running on Foundry. `project.get_openai_client()`
returns an OpenAI-compatible client that routes requests to your hosted container.

Fill in the 2 blanks:
1. The agent name to connect to
2. A weather question to ask

In [ ]:
openai_client = project.get_openai_client(agent_name=___)  # BLANK 9: agent name string

response = openai_client.responses.create(
    input=___,  # BLANK 10: Ask about the weather in a city
)

print("Deployed agent says:")
print(response.output_text)

In [ ]:
# Try another city -- no blanks, just change the question if you want
response2 = openai_client.responses.create(
    input="How about Tokyo? Is it warmer than Paris?",
)

print("Deployed agent says:")
print(response2.output_text)

---
## Verify in the Foundry Portal

Open the [Microsoft Foundry portal](https://ai.azure.com) and confirm:
1. Navigate to your project and open the **Agents** section
2. Your Weather Agent appears in the agents list
3. The status shows **Active**

In [ ]:
print("=" * 55)
print("CHALLENGE 04 -- COMPLETION CHECKLIST")
print("=" * 55)
print()
print("[ ]  Defined a weather tool with typed parameters")
print("[ ]  Wrote agent instructions controlling persona and behavior")
print("[ ]  Assembled main.py from tool + instructions + hosting stack")
print("[ ]  Configured agent.yaml (protocol, resources, env vars)")
print("[ ]  Built container image on ACR")
print("[ ]  Deployed agent via Python SDK")
print("[ ]  Invoked deployed agent -- got weather responses")
print()
print("Show your deployed agent and responses to your coach!")

---
## Key Takeaways

| Concept | Prompt-Based Agents (Ch 01-03) | Hosted Agents (Ch 04) |
|---|---|---|
| **What you write** | System prompt + tool config | Full runtime code (main.py) |
| **Where it runs** | Foundry-managed (serverless) | Your container on Foundry compute |
| **Framework** | Azure AI Agent SDK only | Any framework (Agent Framework, LangGraph, custom) |
| **Protocol** | SDK threads/runs API | HTTP -- Responses or Invocations protocol |
| **Identity** | Shared project identity | Dedicated Entra ID per agent |
| **Deployment** | AgentsClient.create_agent() | SDK create_version() or azd deploy |
| **Best for** | Quick prototyping, simple agents | Production agents, custom logic, multi-framework |

---
## Cleanup

Delete the agent when finished (uncomment to run):
```python
# project.agents.delete(agent_name="weather-agent")
```